# Two-Agent Conversation Patterns

Notebook **05** builds the **reflection pattern**: a **primary** agent drafts work and a **critic** agent reviews it until quality criteria are met. In AutoGen 0.4+ AgentChat, two agents collaborate inside a **`RoundRobinGroupChat`** — they share context, take turns, and stop when **`TextMentionTermination("APPROVE")`** fires.

This notebook covers the **writer/reviewer** workflow for Notes API documentation, **primary/critic system messages**, **`max_turns`**, reading the **message trace** from **`TaskResult`**, combined termination (`MaxMessageTermination | TextMentionTermination`), and **`run_stream`** observation.

**Prerequisites:** **02. Setup and the AgentChat API**, **03. AssistantAgent and ConversableAgent**.

**What you'll learn:**

- How **`RoundRobinGroupChat`** orchestrates two **`AssistantAgent`** instances
- Why **`TextMentionTermination("APPROVE")`** is the critic approval signal
- How to cap dialogue with **`max_turns`** and read **`TaskResult.messages`**
- How the reflection pattern maps to production review loops

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

try:
    import autogen_agentchat
    print("autogen-agentchat:", getattr(autogen_agentchat, "__version__", "installed"))
except ImportError:
    print("autogen-agentchat: pip install autogen-agentchat autogen-ext[openai]")

try:
    import autogen_ext
    print("autogen-ext:", getattr(autogen_ext, "__version__", "installed"))
except ImportError:
    print("autogen-ext: pip install autogen-ext[openai]")

---

## 1. Why Two Agents?

A single agent can draft and self-critique, but separating roles improves quality:

| Single agent | Two-agent reflection |
|--------------|---------------------|
| Same prompt biases both roles | Critic has a distinct system message |
| Easy to skip review | Round-robin forces alternating turns |
| Hard to terminate on approval | **`TextMentionTermination`** stops cleanly |

The **reflection pattern** (primary + critic) is the simplest multi-agent workflow — a stepping stone to larger teams (**08**).

---

## 2. Two-Agent Round-Robin Diagram

```
                    ┌─────────────────────────────┐
                    │   RoundRobinGroupChat       │
                    │   shared message context    │
                    └──────────────┬──────────────┘
                                   │
              ┌────────────────────┴────────────────────┐
              ▼                                         ▼
     ┌─────────────────┐                    ┌─────────────────┐
     │  writer (primary)│  ── turn 1 ──►   │ reviewer (critic)│
     │  drafts content  │  ◄── turn 2 ───   │  feedback/APPROVE│
     └─────────────────┘                    └─────────────────┘
              │                                         │
              └──────────── loop until ─────────────────┘
                         "APPROVE" or max_turns
```

Each agent **broadcasts** its response to the shared context so both see the full transcript.

---

## 3. Notes API Teaching Corpus

We reuse the same documentation chunks (**c1–c5**) from the LangGraph and LangChain tracks so cross-framework comparisons stay consistent:

In [ ]:
NOTES = {
    "n1": "The Notes API is built with FastAPI. Routes live under /notes.",
    "n2": "Run alembic upgrade head after pulling database migrations.",
    "n3": "JWT bearer tokens use Authorization: Bearer header.",
    "n4": "Pytest fixtures for DB setup belong in conftest.py.",
}

DOC_CHUNKS = [
    {"id": "c1", "text": NOTES["n1"]},
    {"id": "c2", "text": NOTES["n2"]},
    {"id": "c3", "text": NOTES["n3"]},
    {"id": "c4", "text": NOTES["n4"]},
    {"id": "c5", "text": "Use httpx.AsyncClient in FastAPI tests for async endpoints."},
]

print("corpus:", [c["id"] for c in DOC_CHUNKS])

---

## 4. Model Client Setup

**`OpenAIChatCompletionClient`** from **`autogen_ext`** is the standard model backend (**02**). Pass **`api_key`** explicitly in notebooks; in production prefer the **`OPENAI_API_KEY`** environment variable.

In [ ]:
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0.3,
)

print("model:", model_client.model)

---

## 5. Primary Agent — Documentation Writer

The **writer** drafts Notes API documentation. Give it a focused system message — no review duties:

In [ ]:
from autogen_agentchat.agents import AssistantAgent

WRITER_SYSTEM = """You are a technical writer for the FastAPI Notes API.
Draft clear, accurate documentation paragraphs.
When the reviewer asks for changes, revise your draft.
Cite documentation chunk ids like [c2] when referencing facts.
Never emit the word APPROVE — only the reviewer approves."""

writer = AssistantAgent(
    name="writer",
    model_client=model_client,
    system_message=WRITER_SYSTEM,
    description="Drafts Notes API documentation.",
)

print("writer:", writer.name)

---

## 6. Critic Agent — Documentation Reviewer

The **reviewer** evaluates drafts for accuracy, clarity, and citations. It must emit **`APPROVE`** (alone or at end of message) when satisfied — this is the termination signal:

In [ ]:
REVIEWER_SYSTEM = """You are a senior documentation reviewer for the Notes API.
Check drafts for accuracy, clarity, and proper chunk citations like [c2].
Give specific, actionable feedback when revisions are needed.
When the draft fully meets quality standards, respond with APPROVE on its own line.
Only the reviewer may say APPROVE."""

reviewer = AssistantAgent(
    name="reviewer",
    model_client=model_client,
    system_message=REVIEWER_SYSTEM,
    description="Reviews documentation drafts and approves when ready.",
)

print("reviewer:", reviewer.name)

---

## 7. `TextMentionTermination("APPROVE")`

**`TextMentionTermination`** stops the team when the target string appears in any agent message. Combine with **`MaxMessageTermination`** using **`|`** (OR) or **`&`** (AND) — see **13. Termination Conditions and Guardrails**.

In [ ]:
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination

approve_termination = TextMentionTermination("APPROVE")
combined_termination = MaxMessageTermination(max_messages=10) | approve_termination

print("approve condition:", type(approve_termination).__name__)
print("combined (OR):", type(combined_termination).__name__)

---

## 8. `RoundRobinGroupChat` with Two Agents

Create the team with participants in speaking order. The first agent speaks first on each new task:

In [ ]:
from autogen_agentchat.teams import RoundRobinGroupChat

doc_team = RoundRobinGroupChat(
    participants=[writer, reviewer],
    termination_condition=approve_termination,
    max_turns=8,
)

print("participants:", [a.name for a in doc_team._participants])
print("max_turns:", doc_team._max_turns)

---

## 9. `max_turns` — Hard Cap on Dialogue

**`max_turns`** limits how many agent turns run before the team stops — a safety rail when the critic never approves. Default is **20**; set lower for teaching demos.

| Parameter | Role |
|-----------|------|
| **`max_turns`** | Hard stop after N agent turns |
| **`termination_condition`** | Soft stop on approval keyword |
| **Both** | Whichever triggers first wins |

---

## 10. Writer/Reviewer Demo — Notes API Docs

Run the team asynchronously. In Jupyter, top-level **`await`** works; in scripts use **`asyncio.run()`** (**02**).

In [ ]:
TASK = (
    "Write a short paragraph explaining how to run database migrations "
    "for the Notes API. Reference the relevant documentation chunks."
)

result = await doc_team.run(task=TASK)
print("stop_reason:", result.stop_reason)
print("message count:", len(result.messages))

---

## 11. Reading the Message Trace

**`TaskResult.messages`** holds the full transcript. Inspect **`source`**, **`content`**, and **`models_usage`** per message:

In [ ]:
from autogen_agentchat.base import TaskResult
from autogen_agentchat.messages import TextMessage


def print_trace(result: TaskResult, max_chars: int = 120) -> None:
    for i, msg in enumerate(result.messages):
        if isinstance(msg, TextMessage):
            preview = (msg.content or "")[:max_chars].replace("\n", " ")
            usage = msg.models_usage
            tokens = f" tokens={usage.prompt_tokens}+{usage.completion_tokens}" if usage else ""
            print(f"{i:02d} [{msg.source}]{tokens}: {preview}...")


print_trace(result)

---

## 12. Find the Approval Turn

Locate which message triggered termination by searching for **`APPROVE`**:

In [ ]:
approval_turns = [
    (i, msg.source)
    for i, msg in enumerate(result.messages)
    if isinstance(msg, TextMessage) and "APPROVE" in (msg.content or "")
]

print("APPROVE found at:", approval_turns)
print("final message source:", result.messages[-1].source)

---

## 13. `run_stream` — Observe Turns Live

**`run_stream`** yields each message as it is produced; the final item is **`TaskResult`**:

In [ ]:
await doc_team.reset()

async for event in doc_team.run_stream(task=TASK):
    if isinstance(event, TaskResult):
        print("\n=== STOP ===", event.stop_reason)
    elif isinstance(event, TextMessage):
        print(f"--- {event.source} ---")
        print((event.content or "")[:200])

---

## 14. `Console` Helper

**`Console`** from **`autogen_agentchat.ui`** formats streaming output for terminal/notebook display (**15**).

In [ ]:
from autogen_agentchat.ui import Console

await doc_team.reset()
# await Console(doc_team.run_stream(task="Draft one sentence about JWT auth for the Notes API."))
print("Uncomment Console(...) above to see formatted streaming output.")

---

## 15. Combined Termination — Budget + Approval

Production teams often combine a message budget with approval:

```python
termination = MaxMessageTermination(max_messages=12) | TextMentionTermination("APPROVE")
```

This stops when the critic approves **or** after 12 messages — whichever comes first.

In [ ]:
budget_team = RoundRobinGroupChat(
    participants=[writer, reviewer],
    termination_condition=MaxMessageTermination(max_messages=6) | TextMentionTermination("APPROVE"),
    max_turns=6,
)

budget_result = await budget_team.run(task="Explain pytest fixture location for the Notes API.")
print("budget stop:", budget_result.stop_reason)
print_trace(budget_result, max_chars=80)

---

## 16. Design Guidelines

| Guideline | Rationale |
|-----------|-----------|
| **Separate system messages** | Writer drafts; reviewer critiques — no role confusion |
| **`APPROVE` is sacred** | Only the critic should emit the termination keyword |
| **Set `max_turns`** | Prevent runaway loops when approval never comes |
| **Read `TaskResult.messages`** | Debug quality issues from the trace |
| **Reset between tasks** | **`await team.reset()`** clears state for a fresh run |
| **Start with two agents** | Master reflection before **08** multi-agent teams |

---

## 17. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Writer says `APPROVE` | Premature termination | Instruct writer never to approve |
| No `max_turns` | Expensive infinite loops | Set 6–12 for review tasks |
| Same system message for both | Weak critique | Distinct reviewer prompt |
| Ignoring `stop_reason` | Can't tell why team stopped | Log **`result.stop_reason`** |
| Reusing team without `reset` | Stale context bleeds in | **`await team.reset()`** before new task |
| Blocking `UserProxyAgent` in long flows | Unstable save/resume | Use only for short approvals (**04**) |

---

## 18. Summary

Two-agent **reflection** is the foundation of AutoGen multi-agent design: a **writer** drafts, a **reviewer** critiques, and **`TextMentionTermination("APPROVE")`** ends the loop when quality is met. **`RoundRobinGroupChat`** shares context and alternates turns; **`max_turns`** adds a hard safety cap.

Key takeaways:

- **`TaskResult.messages`** is your audit trail — inspect **`source`** and content per turn.
- Combine **`MaxMessageTermination | TextMentionTermination`** for budget + approval.
- Use **`run_stream`** or **`Console`** to observe live dialogue (**15**).

Demonstrations set up writer/reviewer agents, ran a Notes API documentation task, printed the message trace, and previewed streaming output.

Next: **06. Tools and Function Registration** — equip **`AssistantAgent`** with async function tools for the Notes API corpus.